In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 🎯 今日学习目标 | 第7周-Day4：多Agent协作模式

> **本周主题：数字员工架构深化**
> **今日主题：从单打独斗到团队协作——多Agent编排的核心模式**


## 📋 今天要掌握的核心概念

| # | 概念 | 关键点 |
|---|------|--------|
| 1 | 主Agent + 子Agent模式 | Orchestrator-Worker 架构 |
| 2 | 任务分发与结果聚合 | 分而治之策略 |
| 3 | TaskFlow工作流 | 多步骤可追踪的持久化任务 |
| 4 | 上下文传递：isolated vs fork | 子Agent的"记忆边界" |
| 5 | Agent间通信协议 | 消息格式、状态同步 |


## 🗺 在架构师路线中的位置


In [ ]:
Level 3 Agent Runtime ← 你在这里
    ├── Agent行为设计（Day1）
    ├── 记忆与语义检索（Day2）
    ├── 任务编排与工作流（Day3）
    ├── 多Agent协作模式（Day4）← 今天
    └── 评估与质量保障（Day5）


💡 **业务关联**：Orchestrator本身就是主Agent，LangChat的各个Capability就是它的子Agent。理解多Agent协作 = 理解Orchestrator的核心运行机制。

# 🔄 往期回顾（W1-W7已学内容速览）


## W1-W2：Transformer与大模型训练
- ✅ 自注意力机制、Multi-Head Attention、位置编码
- ✅ 预训练→SFT→RLHF→DPO→QLoRA微调全链路


## W3：RAG与知识增强
- ✅ 向量检索、Embedding、高级RAG（混合检索、GraphRAG）


## W4：推理与思维链
- ✅ CoT、ToT、DeepSeek R1推理、Prompt工程


## W5-W6：Agent与工具使用实战
- ✅ Function Calling、ReAct模式、Agent安全、框架设计


## W7（本周）
- ✅ Day1：数字员工总览、SOUL.md、输出格式控制
- ✅ Day2：记忆三层结构、MEMORY.md持久化、语义搜索
- ✅ Day3：工具编排（单→多→自动）、Cron定时任务、跨平台消息路由
- 📍 **Day4（今天）：多Agent协作模式**


## 🔗 昨日核心知识点（Day3复习）

| 知识点 | 一句话回顾 |
|--------|-----------|
| 工具编排层级 | 单工具→多工具链→自动化工作流，逐步升级 |
| Cron定时任务 | 周期性执行、延迟提醒、类似Unix crontab |
| 消息路由 | 微信/Telegram/Signal多平台统一处理 |
| 事件驱动 | Agent不仅被动应答，还能主动触发任务 |

# 📚 今日新知识 Part 1：主Agent + 子Agent编排模式


## 1.1 为什么需要多Agent协作？

想象一个公司只有一个员工——他既要接电话、又要写报告、又要做财务。
虽然全能，但效率极低，而且容易出错。

**多Agent协作就是给"数字员工公司"招更多的人：**


In [ ]:
单Agent模式（一个人干所有事）：
用户 → Agent（什么都做）→ 结果

多Agent模式（团队协作）：
用户 → 主Agent（项目经理）
         ├── 子Agent A（数据分析专家）
         ├── 子Agent B（报告撰写专家）
         └── 子Agent C（图表生成专家）
         → 整合结果 → 返回用户


### 🔑 核心概念：Orchestrator-Worker 模式

| 角色 | 类比 | 职责 |
|------|------|------|
| **主Agent（Orchestrator）** | 项目经理 | 理解需求、分解任务、分发、整合结果 |
| **子Agent（Worker）** | 各领域专家 | 执行具体任务、返回专业结果 |
| **消息总线** | 公司内部邮件系统 | Agent间通信、状态同步 |

### 💡 对应到OpenClaw/LangChat


In [ ]:
OpenClaw（OrchestratorAgent = 主Agent）
    ├── sessions_spawn → 子Agent A（搜索任务）
    ├── sessions_spawn → 子Agent B（代码生成）
    └── sessions_spawn → 子Agent C（数据分析）

主Agent负责：
  1. 理解用户意图
  2. 决定需要哪些子Agent
  3. 分发任务
  4. 等待结果
  5. 整合后返回用户


## 1.2 Agent间通信机制

### 三种通信模式


In [ ]:
模式1：管道式（Pipeline）—— 串行传递
Agent A → 结果 → Agent B → 结果 → Agent C → 最终结果
适合：数据处理流水线（查询→分析→报告）

模式2：广播式（Broadcast）—— 并行执行
    ┌→ Agent A → 结果 ┐
主Agent → Agent B → 结果 → 主Agent（聚合）→ 最终结果
    └→ Agent C → 结果 ┘
适合：独立子任务（同时搜索多个数据源）

模式3：对话式（Dialog）—— 协商协作
Agent A ⇄ Agent B（多轮交互，互相补充）
适合：需要反复讨论的复杂任务（代码Review、方案优化）


### 消息格式标准化

```json
{
  "from": "OrchestratorAgent",
  "to": "DataQueryAgent",
  "type": "task_assignment",
  "task_id": "T1",
  "content": "查询2026年6月商管系统销售额",
  "context": {"month": "2026-06", "system": "商管"},
  "deadline": 30,
  "reply_to": "session_abc123"
}


In [ ]:
💡 **对应LangChat**：LangChat的Capability调用本质上就是Agent间通信。knowledge.query → 返回结果 → workflow.execute → 返回结果 → 组合输出。

# 📚 今日新知识 Part 2：TaskFlow 多步骤可追踪工作流

## 2.1 什么是TaskFlow？

如果说主Agent+子Agent是"分工"，那TaskFlow就是"项目管理"。

**TaskFlow = 多步骤、可追踪、可恢复的持久化任务系统**

### 普通任务 vs TaskFlow

| 特性 | 普通任务（sessions_spawn） | TaskFlow |
|------|---------------------------|----------|
| 持续时间 | 几秒到几分钟 | 几分钟到几天 |
| 状态持久化 | ❌ 进程结束就没了 | ✅ 持久化到数据库 |
| 可恢复 | ❌ 失败需重跑 | ✅ 从断点恢复 |
| 可追踪 | ❌ 只看到最终结果 | ✅ 每一步都有状态 |
| 等待外部事件 | ❌ 不支持 | ✅ 支持等待用户回复、API回调 |
| 子任务管理 | ❌ 不支持 | ✅ 支持父子任务层级 |

### TaskFlow 的生命周期


创建（create）→ 等待中（waiting）→ 就绪（ready）→ 执行中（running）
                                                         │
                              ┌──────────────────────────┤
                              │                          │
                        暂停（paused）              失败（failed）
                          │                          │
                          └→ 恢复（running）         └→ 重试（running）
                                                         │
                                                         ▼
                                                   完成（completed）✅


In [ ]:
### 💡 业务场景：商管月度报告生成


TaskFlow: 商管7月经营月报
├── Step 1: 数据收集 [completed]
│   ├── 查询销售额 [done]
│   ├── 查询客流量 [done]
│   └── 查询出租率 [done]
├── Step 2: 数据分析 [completed]
├── Step 3: 报告生成 [running]
│   ├── 生成图表 [done]
│   ├── 撰写分析 [working...]
│   └── 人审确认 [pending]
└── Step 4: 分发 [pending]


In [ ]:
# 📚 今日新知识 Part 3：上下文传递 —— isolated vs fork

## 3.1 子Agent的"记忆边界"问题

> **子Agent需要知道多少主Agent的上下文？**

| 模式 | 类比 | 子Agent知道多少 |
|------|------|----------------|
| **isolated** | "你只需要知道任务本身" | ❌ 不知道项目背景、之前的对话 |
| **fork** | "这是完整的项目背景" | ✅ 知道所有上下文，包括之前的讨论 |

### 什么时候用哪种模式？

| 场景 | 推荐模式 | 理由 |
|------|----------|------|
| 搜索任务 | isolated | 搜索不需要知道对话历史 |
| 代码生成（基于之前讨论） | fork | 需要理解之前的设计讨论 |
| 独立数据处理 | isolated | 只需要输入数据 |
| 代码Review | fork | 需要理解代码的设计意图 |
| 并行探索多个方案 | isolated | 各方案独立，避免相互干扰 |
| 继续之前的工作 | fork | 需要之前的工作成果 |

### 💡 对应OpenClaw


python
# OpenClaw 中的 isolated 模式（默认）
sessions_spawn(
    task="查询商管系统7月销售额TOP10",
    # context 省略 = isolated，子Agent只知道任务本身
)

# OpenClaw 中的 fork 模式
sessions_spawn(
    task="基于上面的讨论，帮我完善这个架构方案",
    context="fork"  # 子Agent获得当前对话的完整上下文
)


In [ ]:
### 💡 对应LangChat

在LangChat的Skill编排中：
- **独立Skill** → isolated（如 data.query 独立执行查询）
- **上下文Skill** → fork（如 report.generate 需要之前步骤的结果）
- 设计Skill管线时，大部分用isolated，只在必要时用fork

# 🔑 英文术语表（10个核心术语）

| # | 英文术语 | 音标 | 中文释义 | 记忆技巧 |
|---|---------|------|---------|---------|
| 1 | **Orchestrator** | /ˈɔːrkɪstreɪtər/ | 编排器、主控Agent | 想象交响乐团指挥（orchestra） |
| 2 | **Sub-agent** | /sʌbˈeɪdʒənt/ | 子代理、子Agent | sub（下级）+ agent（代理） |
| 3 | **Task Decomposition** | /tæsk ˌdiːkɑːmpəˈzɪʃən/ | 任务分解 | de（分开）+ composition（组成） |
| 4 | **Aggregation** | /ˌæɡrɪˈɡeɪʃən/ | 聚合、汇总 | aggregate = 聚集在一起 |
| 5 | **Isolated Context** | /ˈaɪsəleɪtɪd ˈkɑːntekst/ | 隔离上下文 | isolate（隔离）→ 子Agent看不到主Agent历史 |
| 6 | **Fork Context** | /fɔːrk ˈkɑːntekst/ | 分叉上下文 | fork（叉子）→ 从主Agent分叉出完整副本 |
| 7 | **TaskFlow** | /tæsk floʊ/ | 任务流 | task（任务）+ flow（流动） |
| 8 | **Pipeline** | /ˈpaɪplaɪn/ | 管道、流水线 | pipe（管子）+ line（线） |
| 9 | **Broadcast** | /ˈbrɔːdkæst/ | 广播、并行分发 | broad（广泛）+ cast（投掷） |
| 10 | **Checkpoint** | /ˈtʃekpɔɪnt/ | 检查点、断点 | check（检查）+ point（点） |

## 📝 术语造句练习

> "The **Orchestrator** uses **Task Decomposition** to split a complex request into sub-tasks.
> It then **Broadcasts** them to **Sub-agents** running in **Isolated Context**.
> Results are collected through **Aggregation** into a **Pipeline**.
> Each step has a **Checkpoint** for recovery, and the whole process is tracked via **TaskFlow**.

# ✏️ 课堂练习

## 练习1：识别协作模式（选择题）

**场景**：用户要求"帮我查一下三个竞品商城的出租率，然后做个对比分析"

这种任务最适合哪种协作模式？
- A. Pipeline（管道式）
- B. Broadcast（广播式）
- C. Dialog（对话式）

---

## 练习2：选择上下文模式

| 场景 | isolated / fork |
|------|----------------|
| "翻译这段话为英文" | ? |
| "基于我们刚才讨论的架构，画个图" | ? |
| "搜索2026年AI行业报告" | ? |
| "检查我上面写的代码有没有bug" | ? |

---

## 练习3：设计TaskFlow（动手题）

为一个"自动生成商管周报"的需求设计TaskFlow：
1. 从商管系统获取本周数据
2. 与上周数据对比分析
3. 生成图表
4. 人审确认
5. 发送到企业管理群

请写出任务列表（含依赖关系），标注哪些可以并行。

# 📝 课后测试

## Q1（基础题）主Agent的核心职责是什么？
A. 执行所有具体任务
B. 理解需求、分解任务、分发、整合结果
C. 只负责接收用户消息
D. 监控其他Agent的工作状态

## Q2（基础题）以下哪个是 isolated 模式的优点？
A. 上下文完整
B. Token消耗大
C. 干净、安全、Token少
D. 子Agent能理解全部对话历史

## Q3（应用题）你需要设计一个Agent来处理"用户退款"请求，流程包括：
查询订单 → 验证退款条件 → 计算退款金额 → 执行退款 → 发送通知
这个流程最适合用哪种Agent协作模式？为什么？

## Q4（思考题）在LangChat的架构中，以下场景分别对应哪种上下文模式？
- knowledge.query 查询知识库 → ?
- 基于之前查询结果生成报告 → ?
- 并行查询3个不同知识源 → ?
- 对之前生成的报告做Review → ?

## Q5（开放题）如果TaskFlow执行到第4步失败了，系统如何恢复？

---
📌 **提交答案**：直接回复你的答案，AI助手会帮你批改！

# 🎬 推荐学习资源

## 📺 视频教程（国内平台）

### 1. 多智能体协作 Agent开发实战教程
- **平台**: B站（哔哩哔哩）
- **链接**: https://search.bilibili.com/all?keyword=多智能体协作+Agent+开发
- **搜索关键词**: "多Agent协作"、"Multi-Agent LLM"
- **内容**: 多智能体协作架构、任务分发、结果聚合
- **适合**: 理解Orchestrator-Worker模式的实际运行

### 2. LangChain/AutoGen多智能体实战
- **平台**: B站（哔哩哔哩）
- **链接**: https://search.bilibili.com/all?keyword=LangChain+AutoGen+多智能体
- **搜索关键词**: "AutoGen 多Agent"、"LangGraph Agent编排"
- **内容**: AutoGen框架的多Agent对话、LangGraph的状态图工作流
- **适合**: 对比理解OpenClaw的Agent编排设计

## 📖 延伸阅读（国内平台）

### 1. 深入理解多Agent系统架构设计
- **平台**: 知乎
- **链接**: https://www.zhihu.com/search?type=content&q=多Agent协作+LLM+架构设计
- **搜索关键词**: "多Agent协作 LLM"、"Agent编排架构"
- **内容**: 多Agent通信协议、任务分配策略、冲突解决机制

### 2. AutoGen/LangGraph多智能体框架对比与实践
- **平台**: CSDN / 掘金
- **链接**: https://so.csdn.net/so/search?q=多Agent+编排+LangGraph+AutoGen
- **搜索关键词**: "Agent编排 Python实战"
- **内容**: 主流多Agent框架对比、Python代码实战

> ⚠️ **提示**: 平台内容更新频繁，建议直接点击搜索链接，选择2024-2025年发布的最新内容。

# 🔗 知识串联：多Agent协作在架构中的位置

## 从单Agent到多Agent的演进


W5-W6：单Agent + Function Calling → "一个Agent用多个工具"
    ↓
W7 Day1-3：Agent行为 + 记忆 + 工具编排 → "数字员工有了人格和记忆"
    ↓
W7 Day4（今天）：多Agent协作 → "多个数字员工组成团队"
    ↓
W7 Day5（明天）：评估与质量保障 → "怎么确保团队输出质量？"
    ↓
W9：MCP协议 → "Agent之间用什么协议通信？"
    ↓
W10：Agent Runtime进阶 → "OpenClaw vs Hermes运行时设计"


In [ ]:
## 💡 核心洞察：Orchestrator就是"超级主Agent"


OpenClaw = 终极Orchestrator（主Agent）
    ├── Session管理 = 对话上下文管理
    ├── Channel = 多平台消息入口
    ├── Skill = 可插拔的能力（相当于子Agent）
    ├── sessions_spawn = 创建子Agent
    ├── sessions_yield = 等待子Agent完成
    ├── Cron = 定时任务调度
    └── Memory = 持久化记忆
```

**OpenClaw的设计就是多Agent协作的最佳实践！**

# 📝 今日总结


## 🎯 一句话总结
> **多Agent协作 = 主Agent（理解+分发+整合）+ 子Agent（专业执行）+ TaskFlow（可追踪工作流）+ 上下文管理（isolated/fork）**


## 🔑 今日5大收获

| # | 知识点 | 核心要点 |
|---|--------|---------|
| 1 | Orchestrator-Worker模式 | 主Agent管"what"，子Agent管"how" |
| 2 | 三种通信模式 | Pipeline（串行）、Broadcast（并行）、Dialog（协商） |
| 3 | TaskFlow工作流 | 持久化、可追踪、可恢复、支持人审节点 |
| 4 | isolated vs fork | isolated=干净高效，fork=上下文完整 |
| 5 | Agent间消息格式 | from/to/type/content/context标准化 |


## 🚀 明日预告：Day 5 - 评估体系与质量保障
- Agent输出质量怎么评估？
- 三层护栏设计：Prompt护栏 → 工具护栏 → 流程护栏
- 监控与日志：审计轨迹、异常检测、人工介入


## 💡 思考题（选做）
> 如果你要为LangChat设计一个多Agent协作场景，处理"用户问：帮我分析这个商场近半年的经营状况"，你会怎么设计Agent团队？
